In [ ]:
import os
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from Bio import Entrez
import requests
import time

from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_anthropic import ChatAnthropic
from langchain.schema import HumanMessage, SystemMessage

import faiss
from rank_bm25 import BM25Okapi

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")

Entrez.email = "emeshieobim@gmail.com"

In [ ]:
df = pd.read_csv("sample_responses.csv")
queries = df["input"].tolist()[:32]
print(f"✅ Loaded {len(queries)} queries")
print(queries[0])

In [ ]:
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0, api_key=OPENAI_API_KEY)

def transform_to_pubmed_query(natural_query: str) -> str:
    """Transform a natural language medical query into a PubMed search string."""
    system_prompt = """You are a biomedical search expert. 
    Convert the user's natural language medical question into a structured PubMed search string.
    Use MeSH terms where possible. Use AND/OR operators. Keep it under 20 words.
    Return ONLY the search string, nothing else."""
    
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=natural_query)
    ]
    
    response = llm.invoke(messages)
    return response.content.strip()

test_query = queries[0]
pubmed_query = transform_to_pubmed_query(test_query)
print(f"Original:  {test_query}")
print(f"PubMed:    {pubmed_query}")